# US-005：ICA 伪迹去除

**目标：** 用 ICA 识别并去除眼电、心电等伪迹，获得干净脑电信号。

**关键步骤：** 预处理 → ICA 拟合 → 成分识别 → 去除 → 重建

## 1. ICA 原理速览

ICA (Independent Component Analysis) 假设观测信号是若干独立源的线性混合：
$$ X = A \cdot S $$
- $X$：观测 EEG（通道 × 时间）
- $S$：独立源（成分 × 时间）
- $A$：混合矩阵（通道 × 成分）

ICA 的目标是找到解混矩阵 $W = A^{-1}$，使得 $S = W \cdot X$

**去眼电逻辑：** 找到眼电成分 → 把对应列从 A 中删掉 → 用剩余成分重建信号。

## 2. 准备数据（含滤波预处理）

In [ ]:
import mne
import numpy as np

# 加载 sample 数据
sample_dir = mne.datasets.sample.data_path()
raw_fname = sample_dir / "MEG" / "sample" / "sample_audvis_raw.fif"
raw = mne.io.read_raw_fif(raw_fname, preload=True)

# 保留 EEG + EOG 通道（EOG 用于帮助识别眼电成分）
raw.pick_types(eeg=True, eog=True)
print(f"通道: {raw.ch_names}")

# 预处理：1Hz 高通（ICA 要求去掉慢漂移）
raw.filter(l_freq=1.0, h_freq=None)
print(f"高通滤波: {raw.info['highpass']} Hz")

## 3. 拟合 ICA

**重要：** ICA 前必须做高通滤波（≥1 Hz），否则慢漂移会占据独立成分。

In [ ]:
from mne.preprocessing import ICA

# 设置随机种子保证可复现
ica = ICA(
    n_components=20,        # 选择 20 个成分
    method='fastica',       # FastICA 算法（最常用）
    random_state=42,
    max_iter='auto',
)

# 拟合（只对 EEG 通道做 ICA，排除 EOG）
eeg_picks = mne.pick_types(raw.info, eeg=True, eog=False)
ica.fit(raw, picks=eeg_picks)
print(ica)

## 4. 识别眼电成分

### 方法一：看地形图（Topomap）
眼电成分的地形图特征：能量集中在前额，左右对称分布。

In [ ]:
# 绘制所有成分的地形图
ica.plot_components()

### 方法二：看时序图
眼电成分的时序特征：突发性大幅跳变（眨眼）。

In [ ]:
# 绘制成分时序图（可以缩放查看）
ica.plot_sources(raw)

### 方法三：自动检测（用 EOG 通道相关性）
最省事的方式——让 MNE 自动找出和 EOG 通道高相关的成分。

In [ ]:
# 自动找到与 EOG 通道相关的成分
eog_indices, eog_scores = ica.find_bads_eog(
    raw, ch_name='EEG 061',  # sample 数据的 EOG 通道名，实际换成你的 EOG 通道
    threshold=3.0            # Z-score 阈值
)
print(f"眼电成分索引: {eog_indices}")
print(f"得分: {eog_scores[:5]}")

### 方法四：手动指定（凭经验判断）
看完地形图和时序图后，手动标记眼电成分。

In [ ]:
# 假设第 0、1 个成分是眼电
ica.exclude = [0, 1]
print(f"已排除成分: {ica.exclude}")

## 5. 应用 ICA 去除伪迹

`ica.apply()` 从原始信号中减去被排除的成分。

In [ ]:
# 使用自动检测的结果
ica.exclude = eog_indices[:2] if len(eog_indices) >= 2 else eog_indices

# 应用到数据（副本）
raw_clean = raw.copy()
ica.apply(raw_clean)

print(f"ICA 应用完成，排除了 {len(ica.exclude)} 个成分")

## 6. 对比 ICA 前后

肉眼检查去眼电效果。

In [ ]:
# 选择一段含眨眼的区间对比
# 交互式对比
# raw.plot(duration=10, n_channels=10, scalings='auto')
# raw_clean.plot(duration=10, n_channels=10, scalings='auto')

print("ICA 前后对比：取消上方注释查看交互式波形。")

## 7. 保存 ICA 解

ICA 拟合很耗时，建议保存下来以后复用。

In [ ]:
# 保存 ICA 解
ica.save('ica_solution.fif')
print("ICA 解已保存: ica_solution.fif")

# 也保存处理后的数据
# raw_clean.save('eeg_ica_cleaned.fif', overwrite=True)
print("也建议保存处理后的 raw_clean")

## 8. ICA 参数的调优

| 参数 | 含义 | 建议 |
|------|------|------|
| `n_components` | 要分解的成分数 | 通常设为通道数的 80%-100%，或 20-30 |
| `method` | ICA 算法 | 'fastica'（默认/最稳）、'picard'（更快）、'infomax' |
| `random_state` | 随机种子 | 固定值保证可复现 |
| `max_iter` | 最大迭代次数 | 'auto' 通常够，不收敛时加大 |

## 9. 其他伪迹

### 心电 (ECG) 伪迹
```python
ecg_indices, ecg_scores = ica.find_bads_ecg(raw, ch_name='ECG001')
```

### 肌电 (EMG) 伪迹
没有自动化方法，靠地形图（局部、高频分布）和频谱（>20 Hz 功率高）手动判断。

## 10. 练习

1. 对比 `n_components=10` 和 `n_components=30` 的地形图差异
2. 尝试用 `method='picard'`，对比和 FastICA 的速度和结果
3. 手选 ICA 成分 vs 自动检测，看看区别在哪里